# v30 — Isu #2 (v2): Whipsaw/Choppy Awareness sbg Pengali Ketat Filter S/R

**Konteks**: v29 mencoba regime awareness (ADX H1/M15, Trending vs Ranging) sbg pengali ketat
filter S/R v28 -- hasilnya perbaikan TIPIS (PF full-period 1.25->1.31), krn v29 Section 4b
membuktikan performa v28 per regime ADX itu HAMPIR SAMA (Ranging PF 0.68 vs Trending PF 0.58 di
TRAIN) -- ADX rendah/tinggi BUKAN pembeda "bahaya" yang kuat.

**User klarifikasi**: "sideways yang bahaya" itu maksudnya BUKAN semua ranging (ADX rendah),
tapi spesifik kondisi **choppy/whipsaw** -- harga bolak-balik cepat tanpa pola jelas. Ini beda
dari definisi ADX rendah biasa (ADX rendah bisa juga "tenang"/trending pelan, bukan whipsaw).

**Sudah pernah diukur di v20**: `whipsaw_score` = proporsi candle dlm window 12 (1 jam M5)
yang arahnya BEDA dari candle sebelumnya (makin tinggi, makin sering ganti arah/whipsaw). v20
nemuin `Ranging-Choppy` (ADX rendah + whipsaw tinggi) itu PALING LEMAH dari 4 kategori regime
(PF 2.24) walau TETAP untung -- beda dari `Ranging-Tenang` (ADX rendah + whipsaw rendah) yang
malah PALING KUAT (PF 7.32).

**v30 ini**: ulang pola v29 (pengali ketat filter S/R basis v28), TAPI regime-nya diganti dari
"ADX rendah" jadi "whipsaw tinggi" (choppy) -- lebih presisi menyasar kondisi yang TERBUKTI
paling lemah (dari v20), bukan menyamaratakan semua ranging.

**Metodologi**: sama persis v28/v29 -- TRAIN (2019-2023)/TEST (2024-2026) walk-forward, spread
real 1.82, v12_score ASLI + OB filter + H1 alignment + filter S/R v28 sbg BASIS. whipsaw_score
dihitung di M5 (bukan H1/M15 -- whipsaw itu soal harga bolak-balik CEPAT, lebih relevan diukur
di timeframe kecil, beda dari deteksi trend/S/R yg lebih bermakna di TF lebih tinggi).

**TIDAK ADA perubahan ke `usecase.py`** sampai divalidasi & disetujui.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STRATEGY_NAME = "m5_scalping"
VERSION = "v30"

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed" / STRATEGY_NAME
EXPORT_DIR = PROJECT_ROOT / "dataset" / "exports" / STRATEGY_NAME / VERSION
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
(PROCESSED_DIR / VERSION).mkdir(parents=True, exist_ok=True)

INITIAL_EQUITY = 100.0
RISK_PCT = 0.01
CONTRACT_SIZE = 100.0
MIN_LOT = 0.01
LOT_STEP = 0.01
REAL_SPREAD = 1.82

MIN_SAMPLE_TRAIN = 30
MIN_SAMPLE_TEST = 15

pd.set_option("display.width", 180)
plt.rcParams["figure.figsize"] = (14, 5)

## 1. Load cache v28 (skor v12 + OB + H1 EMA + S/R H1/M15) + hitung whipsaw_score M5

In [2]:
WHIPSAW_CACHE_PATH = PROCESSED_DIR / VERSION / "df_2019_2026_whipsaw.parquet"

if WHIPSAW_CACHE_PATH.exists():
    print(f"Load dari cache: {WHIPSAW_CACHE_PATH}")
    df = pd.read_parquet(WHIPSAW_CACHE_PATH)
else:
    print("Belum ada cache -- load v28 base + hitung whipsaw_score M5...")
    v28_cache = PROCESSED_DIR / "v28" / "df_2019_2026_sr.parquet"
    assert v28_cache.exists(), "Cache v28 belum ada"
    df = pd.read_parquet(v28_cache)

    # Sama persis definisi v20: proporsi candle dlm window 12 (1 jam M5) yang arahnya BEDA
    # dari candle sebelumnya -- makin tinggi, makin sering ganti arah (choppy/whipsaw).
    candle_dir = np.sign(df["close"].diff())
    dir_change = (candle_dir != candle_dir.shift(1)).astype(float)
    df["whipsaw_score"] = dir_change.rolling(12, min_periods=6).mean()

    df.to_parquet(WHIPSAW_CACHE_PATH, index=False)
    print(f"Tersimpan ke cache: {WHIPSAW_CACHE_PATH}")

print(f"\nTotal candle: {len(df)}, {df['datetime'].min()} -> {df['datetime'].max()}")
print(f"whipsaw_score stats:\n{df['whipsaw_score'].describe()}")

Belum ada cache -- load v28 base + hitung whipsaw_score M5...


Tersimpan ke cache: D:\Projects\robot-scalping\dataset\processed\m5_scalping\v30\df_2019_2026_whipsaw.parquet

Total candle: 518403, 2019-01-01 23:00:00+00:00 -> 2026-08-06 12:35:00+00:00
whipsaw_score stats:
count    518398.000000
mean          0.519733
std           0.142537
min           0.000000
25%           0.416667
50%           0.500000
75%           0.583333
max           1.000000
Name: whipsaw_score, dtype: float64


## 2. Klasifikasi CHOPPY -- whipsaw_score TINGGI (threshold dari TRAIN, no-leak ke TEST)

In [3]:
TRAIN_END = pd.Timestamp("2024-01-01", tz="UTC")
df_train_for_threshold = df[df["datetime"] < TRAIN_END]
WHIPSAW_HIGH_THRESHOLD = df_train_for_threshold["whipsaw_score"].median()
print(f"Whipsaw threshold (median dari TRAIN saja, no-leak): {WHIPSAW_HIGH_THRESHOLD:.3f}")

df["is_choppy"] = df["whipsaw_score"] >= WHIPSAW_HIGH_THRESHOLD
print(f"\n=== Distribusi CHOPPY vs TENANG (seluruh 2019-2026) ===")
print(df["is_choppy"].value_counts())
print((df["is_choppy"].value_counts(normalize=True) * 100).round(1))

Whipsaw threshold (median dari TRAIN saja, no-leak): 0.500

=== Distribusi CHOPPY vs TENANG (seluruh 2019-2026) ===
is_choppy
True     345047
False    173356
Name: count, dtype: int64
is_choppy
True     66.6
False    33.4
Name: proportion, dtype: float64


## 3. Backtest engine v30: v13 + filter S/R (basis v28) + PENGALI ketat saat CHOPPY

In [4]:
def check_h1_alignment_v30(h1_ema_50, h1_ema_200, direction: str) -> bool:
    if h1_ema_50 is None or h1_ema_200 is None or not np.isfinite(h1_ema_50) or not np.isfinite(h1_ema_200):
        return True
    h1_trend = "UP" if h1_ema_50 > h1_ema_200 else ("DOWN" if h1_ema_50 < h1_ema_200 else "FLAT")
    if direction == "BUY" and h1_trend == "DOWN":
        return False
    if direction == "SELL" and h1_trend == "UP":
        return False
    return True


def run_backtest_v30(
    df_signals: pd.DataFrame,
    adx_min: float = 18.0,
    min_signal_score: float = 9.0,
    sl_mult: float = 2.0,
    tp_mult: float = 4.0,
    max_hold: int = 12,
    sr_near_atr_mult_calm: float = 3.0,
    sr_near_atr_mult_choppy: float = 3.0,
    sr_strong_score_bonus_calm: float = 4.0,
    sr_strong_score_bonus_choppy: float = 4.0,
    sr_min_atr_for_breakout_calm: float = 2.1,
    sr_min_atr_for_breakout_choppy: float = 2.1,
    skip_all_when_choppy: bool = False,  # kalau True, SKIP TOTAL saat choppy (opsi paling ekstrem)
    require_ob_filter: bool = True,
    require_h1_alignment: bool = True,
    spread_points: float = REAL_SPREAD,
    use_fixed_lot: bool = False,
    fixed_lot_value: float = MIN_LOT,
    initial_equity_override: float = None,
) -> pd.DataFrame:
    close_arr = df_signals["close"].to_numpy()
    high_arr = df_signals["high"].to_numpy()
    low_arr = df_signals["low"].to_numpy()
    adx_arr = df_signals["adx"].to_numpy()
    atr_arr = df_signals["atr"].to_numpy()
    score_arr = df_signals["v12_score"].to_numpy()
    ob_bull_arr = df_signals["ob_bull"].to_numpy()
    ob_bear_arr = df_signals["ob_bear"].to_numpy()
    h1_ob_bull_arr = df_signals["h1_ob_bull"].to_numpy()
    h1_ob_bear_arr = df_signals["h1_ob_bear"].to_numpy()
    h1_ema_50_arr = df_signals["h1_ema_50"].to_numpy()
    h1_ema_200_arr = df_signals["h1_ema_200"].to_numpy()
    h1_res_arr = df_signals["h1_sr_resistance"].to_numpy()
    h1_sup_arr = df_signals["h1_sr_support"].to_numpy()
    m15_res_arr = df_signals["m15_sr_resistance"].to_numpy()
    m15_sup_arr = df_signals["m15_sr_support"].to_numpy()
    is_choppy_arr = df_signals["is_choppy"].to_numpy()
    datetime_arr = df_signals["datetime"].to_numpy()
    n = len(df_signals)

    trades = []
    base_equity = initial_equity_override if initial_equity_override is not None else INITIAL_EQUITY
    equity = base_equity
    i = 0
    while i < n:
        adx, atr, close, score = adx_arr[i], atr_arr[i], close_arr[i], score_arr[i]
        if not np.isfinite(atr) or atr <= 0 or not np.isfinite(adx) or not np.isfinite(score):
            i += 1
            continue
        if adx < adx_min:
            i += 1
            continue

        is_choppy = bool(is_choppy_arr[i])
        if is_choppy and skip_all_when_choppy:
            i += 1
            continue

        direction = None
        if score >= min_signal_score:
            direction = "BUY"
        elif score <= -min_signal_score:
            direction = "SELL"
        if direction is None:
            i += 1
            continue

        if require_ob_filter:
            opposing_ob = (
                (direction == "BUY" and (ob_bear_arr[i] > 0 or h1_ob_bear_arr[i] > 0)) or
                (direction == "SELL" and (ob_bull_arr[i] > 0 or h1_ob_bull_arr[i] > 0))
            )
            if opposing_ob:
                i += 1
                continue
        if require_h1_alignment:
            if not check_h1_alignment_v30(h1_ema_50_arr[i], h1_ema_200_arr[i], direction):
                i += 1
                continue

        near_atr_mult = sr_near_atr_mult_choppy if is_choppy else sr_near_atr_mult_calm
        strong_score_bonus = sr_strong_score_bonus_choppy if is_choppy else sr_strong_score_bonus_calm
        min_atr_for_breakout = sr_min_atr_for_breakout_choppy if is_choppy else sr_min_atr_for_breakout_calm

        if direction == "BUY":
            candidates = [v for v in (h1_res_arr[i], m15_res_arr[i]) if np.isfinite(v)]
            opposing_level = min(candidates) if candidates else None
        else:
            candidates = [v for v in (h1_sup_arr[i], m15_sup_arr[i]) if np.isfinite(v)]
            opposing_level = max(candidates) if candidates else None

        if opposing_level is not None:
            dist = abs(opposing_level - close)
            is_near = dist <= (near_atr_mult * atr)
            is_strong_signal = abs(score) >= (min_signal_score + strong_score_bonus)
            is_breakout_atr = atr >= min_atr_for_breakout
            if is_near and not is_strong_signal and not is_breakout_atr:
                i += 1
                continue

        sl_points = sl_mult * atr
        tp_points = tp_mult * atr
        entry_price = close + (spread_points if direction == "BUY" else -spread_points)
        tp_price = entry_price + tp_points if direction == "BUY" else entry_price - tp_points
        sl_price = entry_price - sl_points if direction == "BUY" else entry_price + sl_points

        entry_time = datetime_arr[i]
        exit_price = None
        exit_idx = min(i + max_hold, n - 1)
        window_end = min(i + 1 + max_hold, n)
        for candle_idx in range(i + 1, window_end):
            c_high, c_low = high_arr[candle_idx], low_arr[candle_idx]
            hit_tp = c_high >= tp_price if direction == "BUY" else c_low <= tp_price
            hit_sl = c_low <= sl_price if direction == "BUY" else c_high >= sl_price
            if hit_sl:
                exit_price, exit_time = sl_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
            if hit_tp:
                exit_price, exit_time = tp_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
        if exit_price is None:
            exit_price, exit_time = close_arr[exit_idx], datetime_arr[exit_idx]

        next_i = exit_idx + 1
        price_move = (exit_price - entry_price) if direction == "BUY" else (entry_price - exit_price)
        if use_fixed_lot:
            lot = fixed_lot_value
        else:
            risk_amount = equity * RISK_PCT
            lot = max(round(math.floor((risk_amount / (sl_points * CONTRACT_SIZE)) / LOT_STEP) * LOT_STEP, 2), MIN_LOT) if sl_points > 0 and risk_amount > 0 else MIN_LOT
        pnl = price_move * lot * CONTRACT_SIZE
        equity += pnl
        trades.append({
            "entry_time": entry_time, "direction": direction, "is_choppy": is_choppy, "pnl": pnl,
            "result": "WIN" if pnl > 0 else "LOSS", "equity_after": equity,
        })
        i = next_i

    return pd.DataFrame(trades)


def evaluate(trades: pd.DataFrame, initial_equity: float) -> dict:
    if trades.empty:
        return {"total_trades": 0, "win_rate_pct": 0, "profit_factor": 0, "net_pnl": 0, "max_drawdown_pct": 0}
    wins = trades[trades["pnl"] > 0]
    losses = trades[trades["pnl"] <= 0]
    gross_profit = wins["pnl"].sum()
    gross_loss = losses["pnl"].sum()
    equity_series = pd.Series([initial_equity] + trades["equity_after"].tolist())
    running_max = equity_series.cummax()
    drawdown = (equity_series - running_max) / running_max * 100
    return {
        "total_trades": len(trades),
        "win_rate_pct": round(len(wins) / len(trades) * 100, 2),
        "profit_factor": round(gross_profit / abs(gross_loss), 2) if gross_loss != 0 else float("inf"),
        "net_pnl": round(gross_profit + gross_loss, 2),
        "max_drawdown_pct": round(drawdown.min(), 2),
    }

print("Backtest engine v30 siap.")

Backtest engine v30 siap.


## 4. TRAIN/TEST split & Baseline (v28 murni)

In [5]:
df_train = df[df["datetime"] < TRAIN_END].reset_index(drop=True)
df_test = df[df["datetime"] >= TRAIN_END].reset_index(drop=True)
print(f"TRAIN (2019-2023): {len(df_train)} candle | TEST (2024-2026): {len(df_test)} candle")

trades_base_train = run_backtest_v30(df_train)
trades_base_test = run_backtest_v30(df_test)
baseline_train = evaluate(trades_base_train, INITIAL_EQUITY)
baseline_test = evaluate(trades_base_test, INITIAL_EQUITY)
print("=== Baseline: v28 murni (S/R filter SAMA di choppy vs tenang) ===")
print("TRAIN:", baseline_train)
print("TEST :", baseline_test)

TRAIN (2019-2023): 351136 candle | TEST (2024-2026): 167267 candle


=== Baseline: v28 murni (S/R filter SAMA di choppy vs tenang) ===
TRAIN: {'total_trades': 944, 'win_rate_pct': 23.2, 'profit_factor': np.float64(0.6), 'net_pnl': np.float64(-674.81), 'max_drawdown_pct': np.float64(-674.81)}
TEST : {'total_trades': 810, 'win_rate_pct': 50.12, 'profit_factor': np.float64(1.76), 'net_pnl': np.float64(1741.88), 'max_drawdown_pct': np.float64(-102.96)}


## 4b. Cek pola dulu: apakah v28 memang lebih lemah saat CHOPPY?

In [6]:
print("=== v28 (baseline): performa CHOPPY vs TENANG, TRAIN ===")
for choppy, g in trades_base_train.groupby("is_choppy"):
    label = "CHOPPY" if choppy else "TENANG"
    m = evaluate(g, INITIAL_EQUITY)
    print(f"  {label}: n={m['total_trades']}, win_rate={m['win_rate_pct']}%, PF={m['profit_factor']}, net_pnl=${m['net_pnl']}")

print("\n=== v28 (baseline): performa CHOPPY vs TENANG, TEST ===")
for choppy, g in trades_base_test.groupby("is_choppy"):
    label = "CHOPPY" if choppy else "TENANG"
    m = evaluate(g, INITIAL_EQUITY)
    print(f"  {label}: n={m['total_trades']}, win_rate={m['win_rate_pct']}%, PF={m['profit_factor']}, net_pnl=${m['net_pnl']}")

=== v28 (baseline): performa CHOPPY vs TENANG, TRAIN ===
  TENANG: n=437, win_rate=23.11%, PF=0.64, net_pnl=$-281.62
  CHOPPY: n=507, win_rate=23.27%, PF=0.57, net_pnl=$-393.19

=== v28 (baseline): performa CHOPPY vs TENANG, TEST ===
  TENANG: n=384, win_rate=48.44%, PF=1.97, net_pnl=$1006.45
  CHOPPY: n=426, win_rate=51.64%, PF=1.58, net_pnl=$735.43


## 5. Grid search: pengali ketat filter S/R (+ opsi skip total) KHUSUS saat CHOPPY

In [7]:
import itertools
import time as _time

FIXED_CALM = dict(
    sr_near_atr_mult_calm=3.0, sr_strong_score_bonus_calm=4.0, sr_min_atr_for_breakout_calm=2.1,
)

GRID = {
    "skip_all_when_choppy": [False, True],
    "sr_near_atr_mult_choppy": [3.0, 4.0, 5.0, 6.0],
    "sr_strong_score_bonus_choppy": [4.0, 6.0, 8.0],
    "sr_min_atr_for_breakout_choppy": [2.1, 2.5, 3.0],
}

combos = list(itertools.product(*GRID.values()))
# Kalau skip_all_when_choppy=True, parameter S/R choppy lainnya jadi tidak relevan -- dedup
seen = set()
combos_dedup = []
for c in combos:
    params = dict(zip(GRID.keys(), c))
    if params["skip_all_when_choppy"]:
        key = (True,)
        if key in seen:
            continue
        seen.add(key)
    combos_dedup.append(c)
combos = combos_dedup
print(f"Total kombinasi grid (setelah dedup): {len(combos)}")

t0 = _time.time()
grid_results = []
for idx, combo in enumerate(combos):
    params = dict(zip(GRID.keys(), combo))
    trades = run_backtest_v30(df_train, **FIXED_CALM, **params)
    metrics = evaluate(trades, INITIAL_EQUITY)
    metrics.update(params)
    grid_results.append(metrics)
    if (idx + 1) % 15 == 0:
        print(f"  [{idx+1}/{len(combos)}] {_time.time()-t0:.0f}s")

grid_df = pd.DataFrame(grid_results)
print(f"\nGrid search selesai dalam {_time.time()-t0:.0f}s")

grid_valid = grid_df[grid_df["total_trades"] >= MIN_SAMPLE_TRAIN].sort_values("profit_factor", ascending=False)
print(f"\n=== Top 15 kandidat (sample TRAIN >= {MIN_SAMPLE_TRAIN}) ===")
print(grid_valid.head(15).to_string(index=False))

beating = grid_valid[grid_valid["profit_factor"] > baseline_train["profit_factor"]]
print(f"\nBaseline (v28) TRAIN PF: {baseline_train['profit_factor']}")
print(f"Kandidat mengungguli baseline v28 TRAIN: {len(beating)} dari {len(grid_valid)}")

Total kombinasi grid (setelah dedup): 37


  [15/37] 13s


  [30/37] 27s



Grid search selesai dalam 33s

=== Top 15 kandidat (sample TRAIN >= 30) ===
 total_trades  win_rate_pct  profit_factor  net_pnl  max_drawdown_pct  skip_all_when_choppy  sr_near_atr_mult_choppy  sr_strong_score_bonus_choppy  sr_min_atr_for_breakout_choppy
          479         24.63           0.69  -259.34           -259.34                  True                      3.0                           4.0                             2.1
          811         24.41           0.64  -537.22           -537.22                 False                      6.0                           6.0                             2.1
          813         24.48           0.64  -542.39           -542.39                 False                      6.0                           4.0                             2.1
          811         24.41           0.64  -537.22           -537.22                 False                      6.0                           8.0                             2.1
          836         24.04 

## 6. Validasi TEST out-of-sample

In [8]:
candidates_passing = beating.head(15)
print(f"Kandidat TRAIN mengungguli baseline v28: {len(candidates_passing)}")

if len(candidates_passing) == 0:
    print("\n>>> TIDAK ADA kandidat mengungguli baseline v28 di TRAIN. Validasi TEST DIBATALKAN.")
else:
    test_results = []
    for _, row in candidates_passing.iterrows():
        params = {k: row[k] for k in GRID.keys()}
        trades_test = run_backtest_v30(df_test, **FIXED_CALM, **params)
        m_test = evaluate(trades_test, INITIAL_EQUITY)
        test_results.append({**params, "train_pf": row["profit_factor"], "train_n": row["total_trades"],
                              "test_pf": m_test["profit_factor"], "test_n": m_test["total_trades"],
                              "test_wr": m_test["win_rate_pct"], "test_netpnl": m_test["net_pnl"],
                              "test_maxdd": m_test["max_drawdown_pct"]})

    test_df = pd.DataFrame(test_results)
    print("\n=== Validasi TEST utk kandidat yang menang di TRAIN ===")
    print(test_df.to_string(index=False))

    print(f"\nBaseline (v28) TEST: PF={baseline_test['profit_factor']}, net_pnl={baseline_test['net_pnl']}, n={baseline_test['total_trades']}")

    robust = test_df[(test_df["test_pf"] > baseline_test["profit_factor"]) & (test_df["test_n"] >= MIN_SAMPLE_TEST)]
    print(f"\n>>> Kandidat ROBUST (unggul TRAIN & TEST vs baseline v28, sample TEST>={MIN_SAMPLE_TEST}): {len(robust)}")
    if len(robust) > 0:
        print(robust.to_string(index=False))

Kandidat TRAIN mengungguli baseline v28: 15



=== Validasi TEST utk kandidat yang menang di TRAIN ===
 skip_all_when_choppy  sr_near_atr_mult_choppy  sr_strong_score_bonus_choppy  sr_min_atr_for_breakout_choppy  train_pf  train_n  test_pf  test_n  test_wr  test_netpnl  test_maxdd
                 True                      3.0                           4.0                             2.1      0.69      479     1.95     448    49.55      1100.19      -70.41
                False                      6.0                           6.0                             2.1      0.64      811     1.80     785    51.46      1807.80      -75.01
                False                      6.0                           4.0                             2.1      0.64      813     1.80     785    51.46      1807.80      -75.01
                False                      6.0                           8.0                             2.1      0.64      811     1.80     785    51.46      1807.80      -75.01
                False                      5.0  

## 7. Full-period 2019-2026 comparison (fixed lot, no kill-switch -- basis sama spt v28/v29)

In [9]:
if 'robust' in dir() and len(robust) > 0:
    best = robust.sort_values("test_pf", ascending=False).iloc[0]
    best_params = {k: best[k] for k in GRID.keys()}
    print(f"Kandidat terbaik: {best_params}")

    common_kwargs = dict(use_fixed_lot=True, fixed_lot_value=0.03, initial_equity_override=2000.0)
    trades_v28_full = run_backtest_v30(df, **common_kwargs)
    trades_v30_full = run_backtest_v30(df, **FIXED_CALM, **best_params, **common_kwargs)

    m_v28 = evaluate(trades_v28_full, 2000.0)
    m_v30 = evaluate(trades_v30_full, 2000.0)

    print("\n=== FULL PERIOD 2019-2026: v28 (live sekarang) vs v30 (S/R + pengali whipsaw) ===")
    compare_df = pd.DataFrame([
        {"strategy": "v28 (live)", **m_v28},
        {"strategy": "v30 (whipsaw-aware S/R)", **m_v30},
    ])
    print(compare_df.to_string(index=False))

    def yearly_breakdown(trades, label):
        trades = trades.copy()
        trades["year"] = pd.to_datetime(trades["entry_time"]).dt.year
        rows = []
        for year, g in trades.groupby("year"):
            m = evaluate(g, 2000.0)
            rows.append({"year": year, "strategy": label, **m})
        return pd.DataFrame(rows)

    yearly_v28 = yearly_breakdown(trades_v28_full, "v28")
    yearly_v30 = yearly_breakdown(trades_v30_full, "v30")
    yearly_compare = pd.concat([yearly_v28, yearly_v30]).sort_values(["year", "strategy"])
    print("\n=== Breakdown per tahun ===")
    print(yearly_compare[["year", "strategy", "total_trades", "win_rate_pct", "profit_factor", "net_pnl"]].to_string(index=False))

    years_v30_better = sum(
        1 for year in sorted(yearly_v28["year"].unique())
        if len(yearly_v30[yearly_v30["year"]==year]) and len(yearly_v28[yearly_v28["year"]==year])
        and yearly_v30[yearly_v30["year"]==year]["profit_factor"].values[0] > yearly_v28[yearly_v28["year"]==year]["profit_factor"].values[0]
    )
    print(f"\nv30 mengungguli v28 di {years_v30_better} dari {len(yearly_v28)} tahun")
else:
    print("Tidak ada kandidat robust dari Section 6 -- lewati full-period comparison.")

Kandidat terbaik: {'skip_all_when_choppy': np.True_, 'sr_near_atr_mult_choppy': np.float64(3.0), 'sr_strong_score_bonus_choppy': np.float64(4.0), 'sr_min_atr_for_breakout_choppy': np.float64(2.1)}



=== FULL PERIOD 2019-2026: v28 (live sekarang) vs v30 (S/R + pengali whipsaw) ===
               strategy  total_trades  win_rate_pct  profit_factor  net_pnl  max_drawdown_pct
             v28 (live)          1754         35.63           1.25  2967.73           -116.84
v30 (whipsaw-aware S/R)           927         36.68           1.42  2532.39            -48.46

=== Breakdown per tahun ===
 year strategy  total_trades  win_rate_pct  profit_factor  net_pnl
 2019      v28           176          5.11           0.08  -724.25
 2019      v30            91          4.40           0.09  -389.43
 2020      v28           219         31.96           0.69  -438.09
 2020      v30           100         35.00           0.90   -60.65
 2021      v28           169         23.08           0.72  -251.27
 2021      v30            84         27.38           0.91   -42.67
 2022      v28           172         30.23           0.75  -236.09
 2022      v30            89         32.58           0.90   -45.87
 20

## 8. Kesimpulan

*(diisi setelah lihat hasil eksekusi lengkap Section 3-7 -- placeholder)*